# 23a · Can the agent learn weaker signals if it decides less often, or sees one input? (E23 step 1)

E21 found that Double DQN learns a planted signal at IC 0.4 (sometimes 0.2) but **not at IC 0.1 or 0.05**, while a
simple threshold rule still profits at 0.1. The honest factors sit near 0.1-0.15, so this threshold is the problem.

Two likely reasons, tested here with the same planted signals as E21 (f5 = a noisy copy of the **future** 300-bar
return; a deliberate leak, used only as a positive control):

1. **Decision interval.** At a decision every 5 bars, one 300-bar signal spans 60 decisions, so the reward of a good
   entry is spread over 60 steps of noise. Here the agent decides every **30** or **60** bars (10 or 5 decisions per
   signal), with the same 0.24 % round-trip cost.
2. **Too many inputs.** With 8 factors the agent must find the one that matters. The **single-input** variant keeps
   only the planted f5 (the other 7 factors are set to 0).

| arm | inputs | decision every | IC |
|---|---|---|---|
| 8 factors | cross-fitted factors, f5 planted | 30, 60 bars (5 bars = E21, read from its runs) | 0.1, 0.05 |
| single input | planted f5 only | 5, 60 bars | 0.1, 0.05 |

Double DQN with E16's tuned settings, 1M steps, 2 seeds, greedy policy scored on train and validation every 100k
steps (as E21). Each configuration also gets the best f5 threshold rule on train at the same decision interval.
**If the agent learns IC 0.1 in some configuration, the real factors (IC about 0.13, maybe 0.15 after step 2)
become usable in that configuration.** Test and stress_bear are not loaded.

T4 GPU, 16 runs of about 15-21 min (about 5 h); runs resume after a disconnect. Writes
`thesis_rl/initiation/e23_interval/` and `report/e23a_*`.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "23a_decision_interval")

In [ ]:
import dataclasses, json, time, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from rltrade import factor_rule as FR, learnability as LB, walkforward as WF
from rltrade.env import EnvConfig, Market

INIT = L.root / "initiation"
OUT, REP = INIT / "e23_interval", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"
NEEDED = [f"fixed_plus/market_{s}.npz" for s in ("train", "val")] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)   # test / stress_bear stay zipped
    print("unzipped", NEEDED, "->", LOCAL)
F = INIT / "e20" / "crossfit_mean3"
assert (F / "crossfit.json").exists(), "run notebook 20a first (cross-fitted factors)"
E21 = INIT / "e21_learnability" / "runs"                 # the 8-factor, 5-bar runs come from E21

hp = json.loads((INIT / "agents" / "tuning" / "best_hp.json").read_text())       # E16's tuned Double DQN
AGENT_HP, ENV_OVER = hp["agent_hp"], hp["env"]
AGENT_HP["net_dims"] = tuple(AGENT_HP["net_dims"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS, TOTAL_STEPS, EVAL_EVERY, MIN_TRADES = (0, 1), 1_000_000, 100_000, 5
TRAIN_CHUNKS, VAL_CHUNKS = 32, 16
RHOS = (0.1, 0.05)
CONFIGS = [("8f", g, r) for g in (30, 60) for r in RHOS] + [("single", g, r) for g in (5, 60) for r in RHOS]
KIND = {"8f": "planted", "single": "planted_only"}

VARIANTS, VINFO = {}, {}
for arm in KIND:
    for rho in RHOS:                     # seed 0 = exactly E21's planted signals
        v = f"{arm}_planted_{rho:g}"
        VARIANTS[v], VINFO[v] = LB.write_variant(PREP, F, LOCAL / "e23_factors" / v, KIND[arm], rho=rho, seed=0)
ic_df = pd.DataFrame([{"variant": v, "IC train": i["ic"]["train"], "IC val": i["ic"]["val"]} for v, i in VINFO.items()])
print(f"planted f{LB.PLANT_FACTOR} vs the {LB.PLANT_H}-bar forward return (Spearman, non-overlapping):")
print(ic_df.round(3).to_string(index=False))
print("agent:", AGENT_HP, ENV_OVER, "| device", DEV)

def cfg_for(gap): return EnvConfig(state="A", **{**ENV_OVER, "step_gap": gap})
def run_name(arm, gap, rho, seed): return f"{arm}_gap{gap}_ic{rho:g}_s{seed}"

## 1 · References: buy and hold and the best f5 rule on train, at each decision interval

In [ ]:
PRICE = {s: np.load(PREP / f"market_{s}.npz")["midpoint"].astype(np.float64) for s in ("train", "val")}
ref_rows, BH = [], {}
for gap in sorted({g for _, g, _ in CONFIGS} | {5}):
    cfg = cfg_for(gap)
    m0 = {s: Market(PREP, F, s, cfg, DEV) for s in ("train", "val")}
    BH[gap] = {s: LB.reference_scores(m0[s], cfg, 1)["sharpe"] for s in ("train", "val")}
    for rho in RHOS:
        d = VARIANTS[f"8f_planted_{rho:g}"]
        fac = np.load(Path(d) / "factors_train.npy")
        grid = [g for g in FR.make_grid(FR.signals(fac)) if g["j"] == LB.PLANT_FACTOR]
        tr = FR.run_rules(grid, Market(PREP, d, "train", cfg, DEV), cfg)
        va = FR.run_rules(grid, Market(PREP, d, "val", cfg, DEV), cfg)
        i = WF.select_rule(tr, MIN_TRADES); g = grid[i]
        ref_rows.append({"gap": gap, "IC": rho, "best f5 rule on train": f"q{g['q']} {g['mode']} hold>={g['min_hold']}",
                         "rule train Sharpe": tr["sharpe"][i], "rule val Sharpe": va["sharpe"][i],
                         "rule val return %": va["cum_return_pct"][i], "rule val trades": int(va["num_trades"][i])})
ref_df = pd.DataFrame(ref_rows)
ref_df.to_csv(OUT / "rule_reference.csv", index=False)
print("buy and hold Sharpe (exact):", {g: {s: round(v, 2) for s, v in b.items()} for g, b in BH.items()})
print(ref_df.round(3).to_string(index=False))

## 2 · Train (about 15-21 min per run; resumes)

In [ ]:
t0 = time.time()
for seed in SEEDS:                      # every configuration for seed 0 first: a partial run is already informative
    for arm, gap, rho in CONFIGS:
        LB.train_run(run_name(arm, gap, rho, seed), PREP, VARIANTS[f"{arm}_planted_{rho:g}"], OUT, seed, AGENT_HP,
                     {**ENV_OVER, "step_gap": gap}, TOTAL_STEPS, EVAL_EVERY, device=DEV,
                     train_chunks=TRAIN_CHUNKS, val_chunks=VAL_CHUNKS)
print(f"training done ({(time.time() - t0) / 60:.0f} min this session)")

## 3 · Results: which configurations learn IC 0.1 or 0.05?

In [ ]:
rows = []
def add(arm, gap, rho, seed, path, source):
    if not path.exists(): return
    r = json.loads(path.read_text()); c = pd.DataFrame(r["curve"]); fv, ft = r["final"]["val"], r["final"]["train"]
    rows.append({"inputs": arm, "gap": gap, "IC": rho, "seed": seed, "source": source,
                 "train Sharpe final": ft["sharpe"], "val Sharpe final": fv["sharpe"], "val Sharpe best": c.val_sharpe.max(),
                 "val return % final": fv["cum_return_pct"], "val timing %": fv["attr_timing_pct"],
                 "val trades": fv["num_trades"], "val in market %": fv["exposure_pct"], "val daily timing IC": fv["daily_ic"]})
for arm, gap, rho in CONFIGS:
    for s in SEEDS: add(arm, gap, rho, s, OUT / "runs" / run_name(arm, gap, rho, s) / "result.json", "E23a")
for rho in RHOS:                        # E21: 8 factors, a decision every 5 bars
    for s in SEEDS: add("8f", 5, rho, s, E21 / f"planted_{rho:g}_s{s}" / "result.json", "E21")
runs = pd.DataFrame(rows)
runs.to_csv(OUT / "runs_summary.csv", index=False)
cfg_df = (runs.drop(columns=["seed", "source"]).groupby(["inputs", "gap", "IC"]).mean().reset_index()
          .merge(ref_df[["gap", "IC", "rule val Sharpe", "rule val return %"]], on=["gap", "IC"], how="left"))
cfg_df["buy and hold val Sharpe"] = cfg_df.gap.map(lambda g: BH[g]["val"])
cfg_df["agent beats buy and hold"] = cfg_df["val Sharpe final"] > cfg_df["buy and hold val Sharpe"]
print("every run (final = the policy after 1M steps, one exact pass):")
print(runs.round(2).to_string(index=False))
print("\nper configuration, mean over seeds (rule = best f5 rule on train at the same decision interval):")
print(cfg_df.round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, len(RHOS), figsize=(6 * len(RHOS), 4), sharey=True)
for ax, rho in zip(axes, RHOS):
    for arm, col in (("8f", "C0"), ("single", "C1")):
        g = runs[(runs.inputs == arm) & (runs.IC == rho)]
        ax.scatter(g.gap, g["val Sharpe final"], color=col, s=30, alpha=0.8, label=f"agent, {'8 factors' if arm == '8f' else 'single input'} (each seed)")
        m = g.groupby("gap")["val Sharpe final"].mean()
        ax.plot(m.index, m.values, color=col, lw=1.2)
    r = ref_df[ref_df.IC == rho].sort_values("gap")
    ax.plot(r.gap, r["rule val Sharpe"], color="C2", marker="s", ls="--", lw=1, label="best f5 rule on train")
    ax.plot(sorted(BH), [BH[gp]["val"] for gp in sorted(BH)], color="black", ls=":", lw=1, label="buy and hold")
    ax.axhline(0, color="grey", lw=0.6); ax.set_xscale("log"); ax.set_xticks(sorted(BH)); ax.set_xticklabels(sorted(BH))
    ax.set_xlabel("bars between decisions"); ax.set_title(f"planted signal IC {rho}: validation Sharpe", fontsize=10); ax.grid(alpha=0.3)
axes[0].set_ylabel("Sharpe (exact pass)"); axes[-1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(REP / "e23a_interval.png", dpi=130); plt.show()

In [ ]:
(REP / "e23a_decision_interval.md").write_text(
    "# E23a: decision interval and number of inputs vs the weakest learnable signal (train + validation only)\n\n"
    f"Double DQN, state A, E16 hyperparameters, {len(SEEDS)} seeds x {TOTAL_STEPS:,} steps. Planted f{LB.PLANT_FACTOR} = noisy copy of the "
    f"future {LB.PLANT_H}-bar return (deliberate leak, positive control only); 8f = cross-fitted factors with f5 planted, single = "
    "planted f5 only. Decision every 5 bars with 8 factors comes from E21. Test and stress_bear not loaded.\n\n"
    "## Planted signal strength\n\n" + ic_df.round(3).to_markdown(index=False)
    + "\n\n## Per configuration (mean over seeds)\n\n" + cfg_df.round(3).to_markdown(index=False)
    + "\n\n## Rule reference\n\n" + ref_df.round(3).to_markdown(index=False)
    + "\n\n## Every run\n\n" + runs.round(3).to_markdown(index=False) + "\n\nFigure: e23a_interval.png\n", encoding="utf-8")
print("wrote", REP / "e23a_decision_interval.md")

In [ ]:
AUTO_DISCONNECT = True               # release the GPU once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()